In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/processed/whatsapp-20250408-164953-processed.parquet")

df

In [ ]:
import re

# Clean de data nog wat meer

def remove_url(text):
    return re.sub(r"^https?:\/\/.*[\r\n]*", "", text)


df["message"] = df["message"].apply(lambda x: str(x).replace("\n", " "))
df["message"] = df["message"].apply(lambda x: remove_url(x))
df["message"] = df["message"].apply(lambda x: x.lower())

df

In [ ]:
# Tekststukken van ongeveer 200 woorden maken.

from sklearn.utils import shuffle

# Zet alle berichten per persoon aan elkaar vast
grouped = df.groupby('author')['message'].apply(lambda x: ' '.join(str(m) for m in x)).reset_index()

# Split in chunks van 200 woorden
def split_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

chunk_data = []

for _, row in grouped.iterrows():
    chunks = split_text(row['message'], 200)
    for chunk in chunks:
        if len(chunk.split()) > 20:  # Alleen chunks met genoeg woorden
            chunk_data.append({'author': row['author'], 'text': chunk})

In [ ]:
from collections import Counter

# Tel chunks per auteur
author_counts = Counter([row['author'] for row in chunk_data])

# Maak een lijst met alleen auteurs met minimaal 20 chunks
auteurs_min_20 = {auteur for auteur, count in author_counts.items() if count >= 20}

# Filter de chunk_data
chunk_data_filtered = [row for row in chunk_data if row['author'] in auteurs_min_20]

# Eventueel omzetten naar een DataFrame
chunks_df = pd.DataFrame(chunk_data_filtered)

print(chunks_df['author'].value_counts())
print(chunks_df.author.nunique())

In [ ]:
# # het model snapt geen tekst, dus omzetten in een cijfer-representatie

# from sklearn.feature_extraction.text import TfidfVectorizer

# vectorizer = TfidfVectorizer(max_features=1000)
# X = vectorizer.fit_transform(chunks_df['text'])

# X

from sklearn.feature_extraction.text import CountVectorizer

# Vectoriseer met character 3-grams
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
X = vectorizer.fit_transform(chunks_df["text"])

from sklearn.metrics.pairwise import manhattan_distances

distance = manhattan_distances(X, X)
distance.shape, type(distance)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Verdeel in train en test data: 20% testdata & 42 is het antwoord op alles :)
X_train, X_test, y_train, y_test = train_test_split(distance, chunks_df['author'], test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)




| **Metric**   | **Betekenis** |
|--------------|---------------|
| **Precision** | Hoe vaak het model juist is als het zegt "dit is van persoon X". |
| **Recall**    | Hoe goed het model álle stukjes van persoon X weet te herkennen. |
| **F1-score**  | Een soort gemiddelde van precision en recall. |
| **Support**   | Het aantal test-chunks per persoon (dus hoeveel stukjes tekst er per persoon zijn in de testset). |


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# y_test: echte labels
# y_pred: voorspellingen van het model

# confusion matrix laat zien wanneer het model goed en fout voorspelt
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix van auteursvoorspellingen")
plt.show()


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_model = pca.fit_transform(distance)

# 3. Zet in DataFrame
pca_df = pd.DataFrame({
    "x": pca_model[:, 0],
    "y": pca_model[:, 1],
    "author": chunks_df["author"].values
})


In [ ]:
import seaborn as sns

# Plot
plt.figure(figsize=(6, 4))
sns.scatterplot(data=pca_df, x="x", y="y", hue="author", palette="tab10", s=70)
plt.title("PCA visualisatie van schrijfstijl")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import TSNE

# Maak een kopie van de data om niks te overschrijven
X_subset = X
labels = chunks_df['author'].reset_index(drop=True)

distance = manhattan_distances(X, X)
distance.shape, type(distance)

# Voer t-SNE uit
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(distance)

# Stop het in een dataframe voor plotten
tsne_df = pd.DataFrame({
    "x": X_tsne[:, 0],
    "y": X_tsne[:, 1],
    "author": labels
})

# Plot
plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x="x", y="y", hue="author", palette="tab10")
plt.title("t-SNE visualisatie van schrijfstijl")
plt.tight_layout()
plt.show()

Je ziet dat sommige mensen, zoals sprightly-rhinoceros, echt hun eigen ‘cluster’ hebben hun stijl is dus vrij uniek. Anderen, zoals good-natured-ermine en groovy-salmon, liggen wat dichter bij elkaar, wat kan betekenen dat hun manier van typen iets meer op elkaar lijkt.

🔹 PCA (Principal Component Analysis) is als een vergrootglas dat kijkt naar de grootste verschillen in stijl tussen iedereen. Je ziet bijvoorbeeld dat sprightly-rhinoceros sterk afwijkt van de rest – blijkbaar schrijft die echt op z’n eigen manier.

🔸 t-SNE (t-Distributed Stochastic Neighbor Embedding) is meer een ‘vriendenkaartje’: het kijkt naar wie op wie lijkt, en zet die dichter bij elkaar. Daardoor zie je bijvoorbeeld dat sommige schrijfstijlen een beetje samensmelten (groovy-salmon en good-natured-ermine bijvoorbeeld).

In [ ]:
chunks_df['message_length'] = df['message'].astype(str).str.len()
chunks_df.groupby('author')['message_length'].mean().sort_values(ascending=False)


In [ ]:
import emoji

# Functie om aantal emoji's te tellen in een tekst
def count_emojis(text):
    return sum(1 for char in text if char in emoji.EMOJI_DATA)

# Kolom toevoegen aan je dataframe
chunks_df['emoji_count'] = chunks_df['text'].apply(count_emojis)
chunks_df.groupby('author')['emoji_count'].mean().sort_values(ascending=False)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import nltk

nltk.download('stopwords')
from nltk.corpus import stopwords

dutch_stopwords = stopwords.words('dutch')

vectorizer = CountVectorizer(stop_words=dutch_stopwords)
author_messages = df[df['author'] == 'sprightly-rhinoceros']['message'].dropna().astype(str)
X_rhino = vectorizer.fit_transform(author_messages)

word_counts = X_rhino.sum(axis=0).A1
words = vectorizer.get_feature_names_out()
top_rhino = pd.Series(word_counts, index=words).sort_values(ascending=False).head(20)
print(top_rhino)


In [ ]:
rhino = pca_df[
    (pca_df['x'] < 0) & 
    (pca_df['y'] < -300)  
]

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.scatter(rhino["x"], rhino["y"], color="brown", label="sprightly-rhinoceros")
plt.xlabel("x")
plt.ylabel("y")
plt.title("cluster van sprightly-rhinoceros")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
chunks_df["x"] = pca_df["x"]
chunks_df["y"] = pca_df["y"]

rhino = chunks_df[
    (chunks_df["x"] < 0) &
    (chunks_df["y"] < -300)
]

for i, row in rhino.head(5).iterrows():
    print(f"▶ Bericht {i}:\n{row['text']}\n")


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
import pandas as pd

# Vectoriseer
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
X = vectorizer.fit_transform(chunks_df["text"])
X_dense = X.toarray()

pca_features = PCA(n_components=2)
X_pca_features = pca_features.fit_transform(X_dense)

import pandas as pd

# Zet de PCA loadings in een dataframe
loadings = pd.DataFrame(
    pca_features.components_.T,
    columns=['PC1', 'PC2'],
    index=vectorizer.get_feature_names_out()
)

# Top 10 n-grams die het meest bijdragen aan PC1 (absolute waarde)
top_pc1 = loadings['PC1'].abs().sort_values(ascending=False).head(10)
print("Top 10 belangrijkste n-grams voor PC1:")
print(loadings.loc[top_pc1.index])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

pca = PCA(n_components=2)
pca_model = pca.fit_transform(X_dense)

# Zet in DataFrame
pca_df = pd.DataFrame({
    "x": pca_model[:, 0],
    "y": pca_model[:, 1],
    "author": chunks_df["author"].values
})

# Plot
plt.figure(figsize=(6, 4))
sns.scatterplot(data=pca_df, x="x", y="y", hue="author", palette="tab10", s=70)
plt.title("PCA visualisatie op basis van inhoud")
plt.tight_layout()

plt.legend(
    title="Auteur",
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0.
)

plt.show()
